In [0]:
#Testing the connection of ADLS
# dbutils.fs.ls("/Volumes/adb_caresync_prithwish/landing/landing-volume/neon_postgres/hospitals/2026-11-07_08:32:31/")

In [0]:
# #Now create df using adls file and display

# df=spark.read.format("parquet").load("/Volumes/adb_caresync_prithwish/landing/landing-volume/neon_postgres/hospitals/2026-11-07_08:32:31/")

# display(df)

Above two are for testing

Create a widget and variable to pass source_system(ex neon_postgres),table_name(ex hospitals),landing_ts(ex dd...) dynamically

In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timeStamp","")
dbutils.widgets.text("source_file_format","")


source_system=dbutils.widgets.get("source_system")
table_name=dbutils.widgets.get("table_name")
landing_timeStamp=dbutils.widgets.get("landing_timeStamp")
source_file_format=dbutils.widgets.get("source_file_format")


In [0]:
# Check whether the landing path exists and contains new files before processing
landing_path_check = f"/Volumes/adb_caresync_prithwish/landing/landing-volume/{source_system}/{table_name}/{landing_timeStamp}/"

try:
    landing_files = [
        f for f in dbutils.fs.ls(landing_path_check)
        if not f.isDir() and f.size > 0 and not f.name.startswith(("_", "."))
    ]
except Exception as e:
    # Path/folder does not exist (or is not accessible)
    print(f"Landing path not found: {landing_path_check}")
    landing_files = []

if len(landing_files) == 0:
    print("No new files arrived")
    # Exit cleanly (outside try/except so the exit is not swallowed); notebook ends without failure
    dbutils.notebook.exit("No new files arrived")
else:
    print(f"{len(landing_files)} new file(s) found in {landing_path_check}. Continuing execution.")
    for f in landing_files:
        print(f" - {f.name} ({f.size} bytes)")

In [0]:
#now create df with parameter

landing_path = f"/Volumes/adb_caresync_prithwish/landing/landing-volume/{source_system}/{table_name}/{landing_timeStamp}/"

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

display(landing_df)

In [0]:
#add current timestamp as insert timestamp to check bronze insertion time and landing timestamp as landing timestamp
from pyspark.sql import functions as F
landing_df_2=landing_df.withColumn("landing_timeStamp",F.lit(landing_timeStamp))\
    .withColumn("insert_timeStamp",F.current_timestamp())
display(landing_df_2)

In [0]:
landing_df_2.write.mode("append").format("delta").saveAsTable(f"adb_caresync_prithwish.bronze.{table_name}")

In [0]:
%sql

select * from adb_caresync_prithwish.bronze.hospitals
where landing_timeStamp='2026-11-07_08:32:31'